# Credit Risk Prediction - Exploratory Data Analysis

## Project Overview

The objective of this project is to develop a machine learning model capable of predicting whether a loan applicant is likely to default on a loan.

This notebook focuses on the exploratory data analysis (EDA) stage, including:

- Understanding the dataset
- Inspecting data quality
- Identifying missing values
- Exploring the target variable
- Analyzing numerical and categorical features
- Detecting potential issues before modeling

Dataset:
Home Credit Default Risk (Kaggle)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
#Here I read the document
application = pd.read_csv("application_train.csv")

In [ ]:
#Let´s see the data first
application.head(10)

In [ ]:
application.sample(5)

In [ ]:
application.shape

The dataset contains approximately 307,000 records and more than 120 variables, representing a high-dimensionality problem.

In [ ]:
#Now I can know how many columns are there, data types and memory usage

application.info()

In [ ]:
# Here, look for:
#negative values where they shouldn't exist, unusual maximums, suspicious minimums.

application.describe().T

In [ ]:
#target variable
application["TARGET"].value_counts()

In [ ]:
application["TARGET"].value_counts(normalize=True)

In [ ]:
#Time to create a plot

fig = px.histogram(
    application,
    x="TARGET",
    color="TARGET",
    title="Distribution of Loan Default"
)

fig.show()

#### The dataset is highly imbalanced, with most applicants successfully repaying their loans. This imbalance must be considered during model development because accuracy alone would be a misleading evaluation metric.

In [ ]:
#  I want to answer: Which variables have the poorest quality?

missing = (
    application
    .isna()
    .sum()
    .sort_values(ascending=False)
)

In [ ]:
missing_percent = (
    missing / len(application) * 100
)

In [ ]:
#  Create a dataframe
missing_df = pd.DataFrame({
    "Missing Values": missing,
    "Percentage": missing_percent
})

In [ ]:
# Check variables with only missing values
missing_df = missing_df[missing_df["Missing Values"] > 0]

missing_df.head(20)

In [ ]:
# Now a graph

fig = px.bar(
    missing_df.head(20),
    y=missing_df.head(20).index,
    x="Percentage",
    orientation="h",
    title="Top 20 Variables with Missing Values"
)

fig.show()

In [ ]:
#  Separate numerical and categorical variables


numerical = application.select_dtypes(
    include=np.number
)

categorical = application.select_dtypes(
    exclude=np.number
)

In [ ]:
print(f"Numerical features: {numerical.shape[1]}")
print(f"Categorical features: {categorical.shape[1]}")

In [ ]:
#  Distribution of numerical variables
#  I will not analyze all 120 variables, but rather the most relevant ones.

key_numeric_features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED"
]

application[key_numeric_features].hist(figsize=(15,8))
plt.tight_layout()
plt.show()

DAYS_BIRTH and DAYS_EMPLOYED are expressed as negative values in days, because they represent the number of days prior to the application date.

In [ ]:
# Categorical variables
# We want to know how many categories exist in important variables.

application["NAME_CONTRACT_TYPE"].value_counts()

In [ ]:
application["CODE_GENDER"].value_counts()

In [ ]:
application["NAME_INCOME_TYPE"].value_counts()

#### Now we want to answer:
### Which variables appear to be most closely related to non-compliance?

In [ ]:
corr = application.corr(numeric_only=True)["TARGET"].sort_values()

In [ ]:
# Show the 10 positive and negative correlations.

corr.tail(10)

In [ ]:
corr.head(10)

##### Key Findings

The dataset contains over 300,000 loan applications and more than 120 features.

The target variable is highly imbalanced, with most applicants successfully repaying their loans.

Several variables contain a high percentage of missing values and will require careful preprocessing.

Both numerical and categorical features are present, requiring different preprocessing strategies.

Preliminary correlation analysis suggests that external risk scores and financial attributes may be useful predictors of loan default.

These findings will guide the preprocessing and feature engineering stages of the project.

# Univariate Analysis

## Business Question

Before building any predictive model, it is important to understand the distribution of the most relevant numerical variables.

The objective of this section is to identify:

- Skewed distributions
- Potential outliers
- Suspicious values
- Variables that may require transformations during preprocessing

These insights will guide feature engineering and model selection in subsequent stages.

In [ ]:
key_numeric_features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED"
]

In [ ]:
#Distributions

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for feature, ax in zip(key_numeric_features, axes.flatten()):
    application[feature].hist(
        bins=40,
        ax=ax
    )

    ax.set_title(feature)

plt.tight_layout()
plt.show()

## Interpretation

The selected financial variables exhibit highly skewed distributions, particularly annual income and credit amount.

Such distributions are expected in financial datasets, where a relatively small proportion of customers possess substantially higher incomes or request significantly larger loans.

These observations suggest that logarithmic transformations or robust scaling techniques may improve model performance during preprocessing.

In [ ]:
# Boxplot is preferred over an historgram

fig, axes = plt.subplots(2,3, figsize=(16,10))

for feature, ax in zip(key_numeric_features, axes.flatten()):

    ax.boxplot(
        application[feature].dropna(),
        vert=False
    )

    ax.set_title(feature)

plt.tight_layout()
plt.show()

## Interpretation

Several numerical variables contain extreme observations.

Although these values may initially appear to be outliers, they are not necessarily erroneous. In financial datasets, high-income customers and large loans naturally exist.

Consequently, outlier removal should be performed cautiously, avoiding the elimination of legitimate high-value clients.

# Categorical Variables

## Business Question

Understanding customer demographics is essential for risk assessment.

This section explores the most relevant categorical variables describing applicants' profiles.

In [ ]:
# Kind of contract
application["NAME_CONTRACT_TYPE"].value_counts()

In [ ]:
# Now a histogram
fig = px.histogram(
    application,
    x="NAME_CONTRACT_TYPE",
    color="NAME_CONTRACT_TYPE",
    title="Loan Contract Types"
)

fig.show()

In [ ]:
# Gender
application["CODE_GENDER"].value_counts()

In [ ]:
# Gender histogram
fig = px.histogram(
    application,
    x="CODE_GENDER",
    color="CODE_GENDER",
    title="Applicant Gender"
)

fig.show()

In [ ]:
#Income type
application["NAME_INCOME_TYPE"].value_counts()

In [ ]:
fig = px.histogram(
    application,
    x="NAME_INCOME_TYPE",
    color="NAME_INCOME_TYPE",
    title="Income Type Distribution"
)

fig.show()

## Interpretation

The applicant population is predominantly composed of working individuals, followed by pensioners and commercial associates.

Understanding these demographic segments will be valuable during feature engineering, as employment status may be associated with repayment behavior.

# Correlation Analysis

## Business Question

Which numerical variables appear to be associated with loan default?

Correlation analysis provides an initial indication of potential predictive features. Although correlation does not imply causation, it offers valuable insights for feature selection.

In [ ]:
#Most negatively correlated variables
correlation = (
    application
    .corr(numeric_only=True)["TARGET"]
    .sort_values()
)

In [ ]:
correlation.head(15)

In [ ]:
# Most positively correlated variables
correlation.tail(15)

In [ ]:
# Let's get professional visualization

corr_target = correlation.drop("TARGET")

top_corr = pd.concat([
    corr_target.head(10),
    corr_target.tail(10)
])

fig = px.bar(
    top_corr,
    orientation="h",
    title="Top Features Correlated with Loan Default"
)

fig.show()

## Interpretation

The analysis reveals that external credit scores (EXT_SOURCE variables) exhibit the strongest negative correlation with default, indicating that higher scores are associated with lower credit risk.

Conversely, variables related to payment delays and financial obligations tend to present positive correlations with loan default.

These findings suggest that both demographic and financial variables will play an important role in predictive modeling.

# Key Findings

The exploratory analysis provided several important insights regarding the Home Credit dataset.

### Main Findings

- The dataset contains more than 300,000 loan applications and over 120 explanatory variables.
- The target variable is highly imbalanced, making evaluation metrics such as ROC-AUC, Precision, Recall, and F1-score more appropriate than accuracy.
- Several variables contain substantial proportions of missing values and will require careful preprocessing.
- Numerical variables exhibit skewed distributions and potential outliers, which should be handled using appropriate preprocessing techniques.
- The dataset contains a mixture of numerical and categorical variables that require different transformation strategies.
- External credit score variables demonstrate the strongest relationship with loan default, suggesting high predictive value.

### Next Steps

The next stage of the project will focus on data preprocessing, including:

- Missing value treatment
- Encoding categorical variables
- Feature scaling
- Feature engineering
- Train-test split
- Construction of reproducible preprocessing pipelines